# KNN Imputation

Instead of filling every missing value with one fixed number (mean/median) or encoder (`SimpleImputer`), `KNNImputer` fills each missing value using the **average of its nearest neighbors** — rows most similar to it across the *other* columns. This notebook compares it directly against mean imputation on the Titanic dataset.

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.impute import KNNImputer,SimpleImputer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import accuracy_score

In [2]:
df = pd.read_csv('train.csv')[['Age','Pclass','Fare','Survived']]

FileNotFoundError: [Errno 2] No such file or directory: 'train.csv'

In [ ]:
df.head()

In [ ]:
df.isnull().mean() * 100

In [ ]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=2)

In [ ]:
X_train.head()

## 2. KNN Imputation

`n_neighbors=3` looks at each row's 3 nearest neighbors (by distance across the *other*, non-missing columns) to fill in a missing value. `weights='distance'` gives closer neighbors more influence on the filled value than farther ones, rather than averaging all 3 equally.

### ⚠️ A Scaling Gotcha Worth Knowing

`KNNImputer` finds "nearest" rows using **Euclidean distance** across all columns — exactly the kind of calculation flagged as scale-sensitive in the earlier scaling notebooks. Here, `Fare` (range roughly 0–512) completely dwarfs `Age` (0–80) and `Pclass` (1–3) in the distance calculation, meaning neighbors are being picked almost entirely based on similar `Fare`, with `Age` and `Pclass` barely factoring in at all. Scaling first (demonstrated further down) is the fix.

In [ ]:
knn = KNNImputer(n_neighbors=3,weights='distance')

X_train_trf = knn.fit_transform(X_train)
X_test_trf = knn.transform(X_test)

In [ ]:
lr = LogisticRegression()

lr.fit(X_train_trf,y_train)

y_pred = lr.predict(X_test_trf)

accuracy_score(y_test,y_pred)

## 3. Baseline Comparison — Simple Mean Imputation

In [ ]:
# Comparision with Simple Imputer --> mean

si = SimpleImputer()

X_train_trf2 = si.fit_transform(X_train)
X_test_trf2 = si.transform(X_test)

In [ ]:
lr = LogisticRegression()

lr.fit(X_train_trf2,y_train)

y_pred2 = lr.predict(X_test_trf2)

accuracy_score(y_test,y_pred2)

## 4. Does Scaling First Change the KNN Result?

Scaling every column to the same range **before** imputing lets all three columns contribute fairly to the distance calculation, which should produce more meaningful neighbors than the unscaled version above.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

knn_scaled = KNNImputer(n_neighbors=3, weights='distance')
X_train_trf3 = knn_scaled.fit_transform(X_train_scaled)
X_test_trf3 = knn_scaled.transform(X_test_scaled)

lr = LogisticRegression()
lr.fit(X_train_trf3, y_train)
y_pred3 = lr.predict(X_test_trf3)
accuracy_score(y_test, y_pred3)


## 5. Comparing All Three Side by Side

In [ ]:
import pandas as pd

results = pd.DataFrame({
    'Method': ['KNN Imputer (unscaled)', 'Simple Imputer (mean)', 'KNN Imputer (scaled first)'],
    'Accuracy': [accuracy_score(y_test, y_pred), accuracy_score(y_test, y_pred2), accuracy_score(y_test, y_pred3)]
})
results


## Summary

| Task | Method |
|---|---|
| Impute using nearest-neighbor averages | `KNNImputer(n_neighbors=k, weights='distance')` |
| Weight closer neighbors more | `weights='distance'` *(vs. `'uniform'`, equal weight)* |
| Avoid one large-scale column dominating distance | scale first, **then** `KNNImputer` |

### Trade-offs vs. `SimpleImputer`
`KNNImputer` can capture relationships `SimpleImputer` can't (a missing `Age` filled based on similar passengers, not just the global average) — but it's slower on large datasets (it has to compute distances to every other row) and, as shown above, its quality depends on the input already being on comparable scales.